- 7b sft training
    - https://github.com/volcengine/verl/blob/main/recipe/retool/run_qwen2_7b_sft.sh
- 真正的影响显存的 是 `data.micro_batch_size_per_gpu=4`
- 长序列训练时启用 sp（序列并行是为了节省显存，并不会加快训练速度）
    ```
    data.max_length=16384 \
    data.train_batch_size=32 \
    ulysses_sequence_parallel_size=4 \
    use_remove_padding=true
    ```

### training details

In [4]:
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained('Qwen/Qwen2.5-7B-Instruct')

In [7]:
tokenizer.eos_token, tokenizer.eos_token_id, tokenizer.encode(tokenizer.eos_token)

('<|im_end|>', 151645, [151645])

- EOS（`<|im_end|>`） 不被mask
    - `[0, 1, 2, ... n-1] -> [1, 2, 3, .. n-1, ?]`
    - 模型自然地学会何时终止输出；
- `data.max_length`: `prompt_ids + response_ids`
    - `response_chat_str = response + tokenizer.eos_token`
        - tokenizer 必须要有 eos_token：否则学不会输出 eos
    - `response_ids_output = tokenizer(response_chat_str, return_tensors="pt", add_special_tokens=False)`
    - `response_ids = response_ids_output["input_ids"][0]`
    - `input_ids = torch.cat((prompt_ids, response_ids), dim=-1)`

### model_merger

- https://verl.readthedocs.io/en/latest/advance/checkpoint.html#convert-fsdp-and-megatron-checkpoints-to-huggingface-format-model
- merge huggingface model and test verl checkpoints from FSDP and Megatron backends.

```
python -m verl.model_merger merge \
    --backend fsdp \
    --local_dir checkpoints/verl_fsdp_gsm8k_examples/qwen2_5_0b5_fsdp_saveload/global_step_1/actor \
    --target_dir /path/to/merged_hf_model
```